# Churn Analysis and Customer Intelligence

## 1. Setup & Data Import

In [137]:
# importing libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
import sqlite3

In [138]:
# Connecting to the SQLite database and list its tables
conn = sqlite3.connect('../data/raw/customer_churn.db')

query = "SELECT name FROM sqlite_master WHERE type='table'"
tables = pd.read_sql(query, conn)
tables

,name
0,db_customer
1,db_subscription
2,db_support


In [139]:
# Loading each table into a separate DataFrame (df_db_customer, etc.)
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")
conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


## 2. Data Cleaning

In [140]:
# Inspecting each table's schema directly via SQL (PRAGMA)
conn = sqlite3.connect('../data/raw/customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable: {table_name}")
    columns = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
    print(columns['name'].tolist())

conn.close()


Table: db_customer
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table: db_subscription
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table: db_support
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [141]:
# Detailed look at customer table: types and missing values
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [142]:
# Inspecting top 5 rows to understand the data
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [143]:
# Inspecting bottom 5 rows to understand the data
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


### Cleaning tasks — customer table
1. Rename `name` → `customer_name`
2. Drop `interests` and `pincode` (mostly/entirely empty)
3. Convert `dob` from text to datetime
4. Standardize `gender` (Men→Male, Women→Female)
5. Fill missing `country` values using a state→country map

In [144]:
# 1. Rename 'name' to 'customer_name' for clarity
df_db_customer = df_db_customer.rename(columns = {'name': 'customer_name'})
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [145]:
# 2. Dropping columns that are empty or not useful for analysis
df_db_customer = df_db_customer.drop(columns = ['interests', 'pincode'])
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


In [146]:
# 3. Converting 'dob' from text to datetime
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     str           
 1   customer_name  21 non-null     str           
 2   country        18 non-null     str           
 3   state          21 non-null     str           
 4   gender         21 non-null     str           
 5   dob            21 non-null     datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 1.1 KB


In [147]:
# Checking what gender values actually exist
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female', 'Women', 'Men']
Length: 4, dtype: str

In [148]:
# 4. Standardizing gender values to Male / Female
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men': 'Male', 'Women': 'Female'})
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female']
Length: 2, dtype: str

In [149]:
# Which rows have a missing country?
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,NaN,Delhi,Female,1988-12-10
8,0015-UOCOJ,maya,NaN,Kathmandu,Female,1985-07-07
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01


In [150]:
# 5. Building a state -> country lookup from rows that already have a country
state_country_map = (
    df_db_customer
    .dropna(subset=['country'])           # keep only rows where country is known
    .set_index('state')['country']        # make state the key, country the value
    .to_dict()                            # convert to a dictionary
)
state_country_map

{'Maharashtra': 'India',
 'Karnataka': 'India',
 'Delhi': 'India',
 'Nagaland': 'India',
 'Meghalaya': 'India',
 'Rajasthan': 'India',
 'Kathmandu': 'Nepal',
 'Uttar Pradesh': 'India',
 'Telangana': 'India'}

In [151]:
# Filling missing countries using the state lookup
df_db_customer['country'] = df_db_customer['country'].fillna(
    df_db_customer['state'].map(state_country_map)
)

In [152]:
# Confirm no missing countries remain
df_db_customer['country'].isna().sum()

np.int64(0)

### Cleaning tasks — subscription table
1. Convert `subscription_start_date`, `renewal_date`, `cancellation_date` from text to datetime

Note: `cancellation_date` and `cancellation_reason` have only 6 non-null values.
These nulls are expected — 15 customers are still subscribed, so they correctly have
no cancellation. Not missing data; leave as-is.

In [153]:
# Converting all date columns to datetime
date_cols = ['subscription_start_date', 'renewal_date', 'cancellation_date']
df_db_subscription[date_cols] = df_db_subscription[date_cols].apply(pd.to_datetime)
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     str           
 1   subscription_start_date  21 non-null     datetime64[us]
 2   subscription_type        21 non-null     str           
 3   renewal_date             21 non-null     datetime64[us]
 4   plan_type                21 non-null     str           
 5   contract_type            21 non-null     str           
 6   cancellation_date        6 non-null      datetime64[us]
 7   cancellation_reason      6 non-null      str           
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[us](3), float64(1), int64(2), str(5)
memory usage: 1.9 KB


### Cleaning tasks — support table
1. Drop `col_1` and `comment` (junk / mostly empty)
2. Convert `complaint_date` from text to datetime

Note: table has 9 rows for 7 customers — 2 customers filed complaints twice.
This duplication is handled later during the merge (feature engineering), not here.

In [154]:
# Inspect the support table
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


In [155]:
# Dropping junk columns
df_db_support = df_db_support.drop(columns = ['col_1', 'comment'])
df_db_support.head()

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28 00:00:00,N,60
1,0003-MKNFE,2024-08-28 00:00:00,Y,10
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20
3,0013-MHZWF,2025-03-18 00:00:00,N,90
4,0013-SMEOE,2024-11-01 00:00:00,N,30


In [156]:
# Converting complaint_date to datetime
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 420.0 bytes


## 3. Feature Engineering & Merge

Create a churn flag from cancellation data, then merge all three tables
into one analysis-ready DataFrame keyed on customerid.

In [157]:
# Create churn_flag: 1 if customer cancelled, 0 if still active
df_db_subscription['churn_flag'] = np.where(
    df_db_subscription['cancellation_date'].notna(), 1, 0
)
df_db_subscription[['cancellation_date', 'churn_flag']].head()

,cancellation_date,churn_flag
0,NaT,0
1,2024-09-10,1
2,NaT,0
3,NaT,0
4,2024-02-28,1


In [ ]:
# Merging all three tables into one, keyed on customerid
df = df_db_subscription.merge(df_db_customer, on = 'customerid', how = 'left')
df = df.merge(df_db_support, on = 'customerid', how = 'left')

df.shape

(21, 21)

### ⚠️ Data integrity check: row count inflated after merge

The subscription table has **21 unique customers**, so a left join should
return **21 rows**. Instead the merge produced **23 rows**.

**Diagnosis:** the support table contains **9 rows for only 7 customers** —
two customers filed complaints more than once. Joining on `customerid`
causes these one-to-many matches to *fan out*, duplicating subscription
rows (21 + 2 = 23).

**Why it matters:** at scale this silently inflates every downstream
metric (revenue, churn counts). Always validate row counts across a join.

In [159]:
# How many unique customers should we have?
print("Subscription unique customers:", df_db_subscription['customerid'].nunique())
print("Support rows:", len(df_db_support))
print("Support unique customers:", df_db_support['customerid'].nunique())

Subscription unique customers: 21
Support rows: 9
Support unique customers: 7


**Fix:** aggregated support to one row per customer — kept each customer's
most recent ticket and added a `complaint_count` feature — before merging.
The merge now correctly returns 21 rows.

In [160]:
# Count how many complaints each customer filed
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')
df_db_support[['customerid', 'complaint_date', 'complaint_count']].head()

,customerid,complaint_date,complaint_count
0,0003-MKNFE,2024-08-28,2
1,0003-MKNFE,2024-08-28,2
2,0013-EXCHZ,2024-01-20,1
3,0013-MHZWF,2025-03-18,1
4,0013-SMEOE,2024-11-01,1


In [ ]:
# Keep each customer's most recent complaint only (one row per customer)
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates(subset = 'customerid', keep='last')
df_db_support.shape

(7, 5)